INTRODUCTION à torch.autograd

BACKGROUND

In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\romai/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [ ]:
prediction = model(data) # forward pass
print(prediction.shape)

torch.Size([1, 1000])


In [ ]:
loss = (prediction - labels).sum()
loss.backward()

In [10]:
#C'est l'algorithme qui va se charger de modifier concrètement les poids de ton réseau pour réduire l'erreur, 
#en appliquant la fameuse formule de la descente de gradient qu'on a décortiquée ensemble au tout début.
#lr : learning rate (taux d'apprentissage)
#momentum : ajoute une forme d'"inertie" à la descente de gradient // permet de traverser les plats de la fonction de loss
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9) 
optim.step() #gradient descent

DIFFERENTIATION IN AUTOGRAD

In [29]:
import torch

a = torch.tensor([2.,3.], requires_grad=True) #obliger des mettres des floats dans les []
b = torch.tensor([6.,4.], requires_grad=True) #requires_grad=True indique qu'il faut préparer des graphes de calculs dynamiques

Q =3*a**3 - b**2

print(a,b,Q)

tensor([2., 3.], requires_grad=True) tensor([6., 4.], requires_grad=True) tensor([-12.,  65.], grad_fn=<SubBackward0>)


In [30]:
external_grad = torch.tensor([1., 1.]) #[ 1,1] permet d'effectuer le .backward() sur chaque composante les 1 après les autres de Q
Q.backward(gradient=external_grad) # les infos de la règle de la chaîne sont stockées sur a.grad et b.grad

In [31]:
# check if collected gradients are correct
print(9*a**2 == a.grad)
print(-2*b == b.grad)

tensor([True, True])
tensor([True, True])


OPTIONAL - VECTOR CALCULUS USING autograd

COMPUTATIONAL GRAPH

EXCLUSION FROM THE DAG

In [32]:
x = torch.rand(5, 5)
y = torch.rand(5, 5)
z = torch.rand((5, 5), requires_grad=True)

a = x + y
print(f"Does `a` require gradients?: {a.requires_grad}")
b = x + z
print(f"Does `b` require gradients?: {b.requires_grad}")

Does `a` require gradients?: False
Does `b` require gradients?: True


In [34]:
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze all the parameters in the network
for param in model.parameters():
    param.requires_grad = False

In [35]:
model.fc = nn.Linear(512, 10)

In [36]:
# Optimize only the classifier
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)